# AHR v2 — batched Scout and Worker search

This notebook implements a fixed Queen batch scheduler, Scout move priors and Worker current-player outcome values in exact Go tree search. Every legal move, including pass, remains eligible. Search estimates do not certify optimal play.

**Inputs:** tagged definitions in `00_contracts_and_recovery.ipynb`, `01_rules_and_replay.ipynb` and `02_scout_worker_network.ipynb`. **Outputs:** a tested search API, a validation receipt and isolated recovery checkpoints. **Expected runtime:** under five minutes on CPU; no training or opponent labels. **Checkpoints:** `checkpoints/ahr-v2-search-validation/`. The final recovery cell shows the exact fresh-process resume procedure; complete training states belong to notebook 00's checkpoint manager.

The returned policy uses completed real visits only. Wall-clock or depth cutoffs preserve `UNKNOWN` for unproved game results and keep move legality separate.

In [ ]:
from pathlib import Path
import json, hashlib, copy, math, time, dataclasses, tempfile, shutil, subprocess, sys, os
import numpy as np

GO2_PROJECT = Path.cwd().resolve()
if not (GO2_PROJECT / 'experiments/ahr_v2/00_contracts_and_recovery.ipynb').exists():
    GO2_PROJECT = Path('/home/benjamin/workspace/ahr_go')
import nbformat
_go2_shared = nbformat.read(GO2_PROJECT / 'experiments/ahr_v2/00_contracts_and_recovery.ipynb', 4)
for _go2_cell in _go2_shared.cells:
    if _go2_cell.cell_type == 'code' and 'ahr-export' in _go2_cell.metadata.get('tags', []):
        exec(compile(_go2_cell.source, '00_contracts_and_recovery.ipynb', 'exec'), globals())
for _go2_name in ('01_rules_and_replay.ipynb', '02_scout_worker_network.ipynb'):
    go2_import_exports(GO2_PROJECT / 'experiments/ahr_v2' / _go2_name, globals())
print('Loaded definitions only; upstream generation and training cells did not run.')

## Search contract

PUCT compares values from each parent player's perspective. A leaf value is negated at every backup edge. Exact terminal scoring runs before neural evaluation. Pending leaf reservations and virtual visits organize each inference batch; they are removed before real visits are committed. Repeated reservations never add another completed visit.

A saved tree binds the exact rules/history, model tensors, source definitions, search settings and future Queen context. The fixed scheduler can be replaced later only under a new context and model identity.

A requested pause takes effect after the next atomic batch, so it can slightly exceed its simulation threshold. The immutable simulation budget and batch shape stay fixed across resume.

### Parameters, identities and Scout expansion

In [ ]:
import dataclasses, copy, math, time
import numpy as np

@dataclasses.dataclass(frozen=True)
class Go2SearchConfig:
    simulations: int = 64
    batch_size: int = 8
    c_puct: float = 1.5
    virtual_loss: float = 1.0
    prior_floor: float = 1e-6
    seed: int = 2026100603
    max_depth: int = 256
    wall_time_seconds: float | None = None
    root_noise_fraction: float = 0.0
    root_dirichlet_alpha: float = 0.3

    def __post_init__(self):
        if self.simulations < 1 or self.batch_size < 1 or self.max_depth < 1:
            raise ValueError('Simulation, batch and depth budgets must be positive.')
        if self.c_puct <= 0 or self.virtual_loss < 0 or self.prior_floor <= 0:
            raise ValueError('Invalid PUCT, reservation or exploration settings.')
        if self.wall_time_seconds is not None and self.wall_time_seconds < 0:
            raise ValueError('Wall-clock budget cannot be negative.')
        if not 0 <= self.root_noise_fraction <= 1 or self.root_dirichlet_alpha <= 0:
            raise ValueError('Invalid root exploration noise settings.')


def _go2_search_game():
    return {
        'legal': go2_legal_actions,
        'step': go2_step,
        'key': go2_cache_key,
        'dump': go2_state_to_dict,
        'load': go2_state_from_dict,
        'terminal': lambda state: bool(state.terminal),
        'value': lambda state: float(go2_outcome(state)['value_current']),
        'actions': lambda state: int(state.size * state.size + 1),
    }


def _go2_search_identity(model, state, config, model_identity, context, project, game):
    if model_identity is None:
        if model is None:
            raise ValueError('A custom evaluator requires an explicit immutable model identity.')
        model_identity = go2_model_identity(model)
    return {
        'schema': 'ahr-go-v2-search-tree-v1',
        'model': str(model_identity),
        'sources': go2_source_identity(go2_project(project)),
        'config': dataclasses.asdict(config),
        'root': go2_digest(game['key'](state)),
        'context': go2_digest({} if context is None else context),
    }


def _go2_node(state, game):
    terminal = game['terminal'](state)
    return {'state': state, 'terminal': terminal, 'expanded': terminal,
            'value': game['value'](state) if terminal else None,
            'visits': 0, 'virtual_visits': 0, 'edges': {}}


def _go2_new_tree(identity, state, config, game):
    rng = np.random.default_rng(config.seed)
    return {'identity': identity, 'config': dataclasses.asdict(config),
            'root': 0, 'next_node': 1, 'nodes': {0: _go2_node(state, game)},
            'cache': {}, 'pending': [], 'rng': rng,
            'completed_simulations': 0, 'elapsed_seconds': 0.0,
            'counters': {'neural_batches': 0, 'neural_positions': 0,
                         'cache_hits': 0, 'duplicate_reservations': 0,
                         'depth_cutoffs': 0, 'terminal_simulations': 0,
                         'atomic_batches': 0},
            'queen': {'kind': 'fixed_batch_schedule', 'batch_size': config.batch_size}}


def _go2_inference_key(tree, state, game):
    return go2_digest({'state_and_rules': game['key'](state),
                       'model': tree['identity']['model'],
                       'context': tree['identity']['context']})


def _go2_expand(tree, node_id, output, config, game):
    node = tree['nodes'][node_id]
    if node['terminal']:
        raise AssertionError('A terminal position cannot require neural inference.')
    prior = np.asarray(output['prior'], dtype=np.float64)
    value = float(output['value'])
    action_count = game['actions'](node['state'])
    legal = sorted(int(action) for action in game['legal'](node['state']))
    if prior.shape != (action_count,) or not np.isfinite(prior).all() or np.any(prior < 0):
        raise ValueError('Scout output must contain finite nonnegative action probabilities.')
    if not math.isfinite(value) or not -1.000001 <= value <= 1.000001:
        raise ValueError('Worker value must be finite and in [-1, 1].')
    if not legal:
        raise AssertionError('A nonterminal position must have legal actions.')
    illegal = np.ones(action_count, dtype=bool)
    illegal[legal] = False
    if np.any(prior[illegal] > 1e-8):
        raise ValueError('Scout probabilities assign mass to an illegal action.')
    masked = np.zeros(action_count, dtype=np.float64)
    masked[legal] = np.maximum(prior[legal], config.prior_floor)
    masked /= masked.sum()
    if node_id == tree['root'] and config.root_noise_fraction:
        noise = tree['rng'].dirichlet(np.full(len(legal), config.root_dirichlet_alpha))
        masked[legal] = ((1-config.root_noise_fraction)*masked[legal]
                         + config.root_noise_fraction*noise)
    node['value'] = max(-1.0, min(1.0, value))
    node['edges'] = {action: {'prior': float(masked[action]), 'n': 0, 'w': 0.0,
                            'virtual_n': 0, 'child': None} for action in legal}
    node['expanded'] = True

### Batch neural evaluation

In [ ]:
def _go2_evaluate_nodes(tree, node_ids, evaluator, config, game):
    missing = {}
    for node_id in node_ids:
        node = tree['nodes'][node_id]
        if node['terminal'] or node['expanded']:
            continue
        key = _go2_inference_key(tree, node['state'], game)
        if key in tree['cache']:
            tree['counters']['cache_hits'] += 1
            _go2_expand(tree, node_id, tree['cache'][key], config, game)
        else:
            missing.setdefault(key, []).append(node_id)
    if not missing:
        return
    keys = list(missing)
    states = [tree['nodes'][missing[key][0]]['state'] for key in keys]
    outputs = evaluator(states)
    if len(outputs) != len(states):
        raise ValueError('Evaluator result count differs from the inference batch.')
    validated = []
    for key, output in zip(keys, outputs):
        if 'network_version' in output and str(output['network_version']) != tree['identity']['model']:
            raise ValueError('Evaluator model identity changed during an active search.')
        minimal = {'prior': np.asarray(output['prior'], dtype=np.float64).tolist(),
                   'value': float(output['value'])}
        # Validate every output on a disposable tree before committing any expansion.
        probe = copy.deepcopy(tree['nodes'][missing[key][0]])
        check_tree = {'nodes': {1: probe}, 'root': 0, 'rng': tree['rng']}
        _go2_expand(check_tree, 1, minimal, config, game)
        validated.append((key, minimal))
    tree['counters']['neural_batches'] += 1
    tree['counters']['neural_positions'] += len(states)
    for key, minimal in validated:
        tree['cache'][key] = minimal
        for node_id in missing[key]:
            _go2_expand(tree, node_id, minimal, config, game)

### Fixed Queen scheduling and backup

In [ ]:
def _go2_select_path(tree, config, game, reserved=()):
    reserved = set(reserved)
    def descend(node_id, path):
        node = tree['nodes'][node_id]
        depth_cutoff = len(path) >= config.max_depth
        if node['terminal'] or not node['expanded'] or depth_cutoff:
            if node_id in reserved and not node['terminal']:
                tree['counters']['duplicate_reservations'] += 1
                return None
            return path, node_id, bool(depth_cutoff and not node['terminal'])
        total = node['visits'] + node['virtual_visits']
        scored = []
        for action, edge in node['edges'].items():
            count = edge['n'] + edge['virtual_n']
            q = ((edge['w'] - config.virtual_loss * edge['virtual_n']) / count
                 if count else 0.0)
            exploration = config.c_puct * edge['prior'] * math.sqrt(1+total) / (1+count)
            scored.append((q + exploration, edge['prior'], -action, action))
        # Reservations exclude a leaf only for this inference batch. They never
        # remove a legal edge from the persistent Scout action set.
        for _, _, _, action in sorted(scored, reverse=True):
            edge = node['edges'][action]
            if edge['child'] is None:
                child = game['step'](node['state'], action)
                child_id = tree['next_node']
                tree['next_node'] += 1
                tree['nodes'][child_id] = _go2_node(child, game)
                edge['child'] = child_id
            found = descend(edge['child'], [*path, (node_id, action)])
            if found is not None:
                return found
        return None
    return descend(tree['root'], [])


def _go2_reserve(tree, path, leaf_id):
    for node_id, action in path:
        tree['nodes'][node_id]['virtual_visits'] += 1
        tree['nodes'][node_id]['edges'][action]['virtual_n'] += 1
    tree['nodes'][leaf_id]['virtual_visits'] += 1


def _go2_release(tree, path, leaf_id):
    for node_id, action in path:
        tree['nodes'][node_id]['virtual_visits'] -= 1
        tree['nodes'][node_id]['edges'][action]['virtual_n'] -= 1
    tree['nodes'][leaf_id]['virtual_visits'] -= 1


def _go2_collect_batch(tree, config, game, count):
    if tree['pending']:
        raise AssertionError('Resolve a pending batch before collecting another.')
    reserved = set()
    while len(tree['pending']) < count:
        selected = _go2_select_path(tree, config, game, reserved)
        if selected is None:
            break
        path, leaf_id, depth_cutoff = selected
        if not tree['nodes'][leaf_id]['terminal']:
            reserved.add(leaf_id)
        _go2_reserve(tree, path, leaf_id)
        tree['pending'].append({'path': path, 'leaf': leaf_id,
                                'depth_cutoff': bool(depth_cutoff),
                                'simulation_id': tree['completed_simulations'] + len(tree['pending'])})
    if not tree['pending']:
        raise AssertionError('A nonterminal root must permit at least one search simulation.')


def _go2_resolve_batch(tree, evaluator, config, game):
    pending = tree['pending']
    if not pending:
        return
    _go2_evaluate_nodes(tree, [record['leaf'] for record in pending], evaluator, config, game)
    # Once evaluation succeeds, this entire batch is an atomic real-visit commit.
    for record in pending:
        path, leaf_id = record['path'], record['leaf']
        leaf = tree['nodes'][leaf_id]
        value = float(leaf['value'])
        _go2_release(tree, path, leaf_id)
        leaf['visits'] += 1
        for node_id, action in reversed(path):
            value = -value
            node = tree['nodes'][node_id]
            node['edges'][action]['n'] += 1
            node['edges'][action]['w'] += value
            node['visits'] += 1
        tree['completed_simulations'] += 1
        tree['counters']['depth_cutoffs'] += int(record['depth_cutoff'])
        tree['counters']['terminal_simulations'] += int(leaf['terminal'])
    tree['pending'] = []
    tree['counters']['atomic_batches'] += 1
    go2_validate_search_tree(tree)

### Visit and reservation validation

In [ ]:
def go2_validate_search_tree(tree):
    nodes = tree['nodes']
    if tree['root'] not in nodes:
        raise ValueError('Search root is absent.')
    simulation_ids = [record['simulation_id'] for record in tree['pending']]
    if len(simulation_ids) != len(set(simulation_ids)):
        raise ValueError('A pending batch double-counts a simulation identity.')
    pending_leaf_ids = [record['leaf'] for record in tree['pending'] if not nodes[record['leaf']]['terminal']]
    if len(pending_leaf_ids) != len(set(pending_leaf_ids)):
        raise ValueError('A pending batch contains duplicate leaf reservations.')
    expected_node_virtual = {node_id: 0 for node_id in nodes}
    expected_edge_virtual = {(node_id, action): 0 for node_id, node in nodes.items()
                             for action in node['edges']}
    for record in tree['pending']:
        node_id = tree['root']
        for parent_id, action in record['path']:
            if parent_id != node_id or action not in nodes[parent_id]['edges']:
                raise ValueError('A pending path is inconsistent with the tree.')
            edge = nodes[parent_id]['edges'][action]
            node_id = edge['child']
            expected_node_virtual[parent_id] += 1
            expected_edge_virtual[(parent_id, action)] += 1
        if node_id != record['leaf'] or node_id not in nodes:
            raise ValueError('A pending leaf is inconsistent with its path.')
        expected_node_virtual[node_id] += 1
    for node_id, node in nodes.items():
        if node['visits'] < 0 or node['virtual_visits'] != expected_node_virtual[node_id]:
            raise ValueError('Search node visit conservation failed.')
        if node['expanded'] and not node['terminal'] and not node['edges']:
            raise ValueError('Expanded nonterminal node has no legal edges.')
        for action, edge in node['edges'].items():
            if edge['n'] < 0 or edge['virtual_n'] != expected_edge_virtual[(node_id, action)]:
                raise ValueError('Search edge reservation conservation failed.')
            if not math.isfinite(edge['w']) or abs(edge['w']) > edge['n'] + 1e-6:
                raise ValueError('Search accumulated values are invalid.')
            if edge['child'] is not None:
                if edge['child'] not in nodes or nodes[edge['child']]['visits'] != edge['n']:
                    raise ValueError('Child visits and parent edge visits differ.')
        if sum(edge['n'] for edge in node['edges'].values()) > node['visits']:
            raise ValueError('Outgoing visits exceed total node visits.')
    root = nodes[tree['root']]
    if root['visits'] != tree['completed_simulations']:
        raise ValueError('Completed simulations and root visits differ.')
    if root['expanded'] and not root['terminal']:
        if sum(edge['n'] for edge in root['edges'].values()) != tree['completed_simulations']:
            raise ValueError('Root policy includes incomplete or duplicate simulations.')
    return True

### Serialization and public search API

In [ ]:
def go2_search_state_dict(tree, _game=None):
    game = _go2_search_game() if _game is None else _game
    go2_validate_search_tree(tree)
    payload = {key: copy.deepcopy(value) for key, value in tree.items() if key not in ('nodes', 'rng')}
    payload['rng'] = copy.deepcopy(tree['rng'].bit_generator.state)
    payload['nodes'] = {}
    for node_id, node in tree['nodes'].items():
        saved = {key: copy.deepcopy(value) for key, value in node.items() if key != 'state'}
        saved['state'] = game['dump'](node['state'])
        payload['nodes'][int(node_id)] = saved
    payload['payload_sha256'] = go2_digest(payload)
    return payload


def go2_search_from_state_dict(payload, model, state, config=None, model_identity=None,
                               context=None, project=None, _game=None):
    game = _go2_search_game() if _game is None else _game
    if _game is None:
        go2_verify_replay(state)
    payload = copy.deepcopy(payload)
    digest = payload.pop('payload_sha256', None)
    if digest is None or digest != go2_digest(payload):
        raise ValueError('Search payload checksum is invalid; refusing to restart.')
    config = Go2SearchConfig(**payload['config']) if config is None else config
    identity = _go2_search_identity(model, state, config, model_identity, context, project, game)
    if payload['identity'] != identity:
        raise ValueError('Search model, source, configuration, rules/history or context changed.')
    rng = np.random.default_rng()
    rng.bit_generator.state = payload.pop('rng')
    nodes = {}
    for node_id, saved in payload.pop('nodes').items():
        saved['state'] = game['load'](saved['state'])
        saved['edges'] = {int(action): edge for action, edge in saved['edges'].items()}
        nodes[int(node_id)] = saved
    tree = {**payload, 'nodes': nodes, 'rng': rng}
    tree['cache'] = {str(key): value for key, value in tree['cache'].items()}
    go2_validate_search_tree(tree)
    return tree


def go2_search(model, state, config=None, tree=None, stop_after_simulations=None,
               on_batch=None, evaluator=None, model_identity=None, context=None,
               project=None, _game=None):
    config = Go2SearchConfig() if config is None else config
    game = _go2_search_game() if _game is None else _game
    identity = _go2_search_identity(model, state, config, model_identity, context, project, game)
    if tree is None:
        if _game is None:
            go2_verify_replay(state)
        tree = _go2_new_tree(identity, state, config, game)
    elif tree['identity'] != identity:
        raise ValueError('Active search identity differs from this request.')
    go2_validate_search_tree(tree)
    root = tree['nodes'][tree['root']]
    if root['terminal']:
        return {'action': None, 'policy_target': [0.0]*game['actions'](state),
                'visit_counts': [0]*game['actions'](state), 'tree': tree,
                'status': 'TERMINAL', 'search_status': 'EXACT_TERMINAL',
                'move_legality': 'NO_MOVE_TERMINAL', 'value': root['value'],
                'completed_simulations': 0, 'counters': copy.deepcopy(tree['counters'])}
    if evaluator is None:
        if model is None:
            raise ValueError('Provide a model or an explicit evaluator.')
        evaluator = lambda states: go2_eval_batch(model, states)
    started = time.monotonic()
    stop = config.simulations if stop_after_simulations is None else min(config.simulations, int(stop_after_simulations))
    if stop < tree['completed_simulations']:
        raise ValueError('Resume stop position precedes completed simulations.')
    budget_cutoff = False
    try:
        if tree['pending']:
            _go2_resolve_batch(tree, evaluator, config, game)
            tree['elapsed_seconds'] += time.monotonic() - started
            started = time.monotonic()
            if on_batch is not None:
                on_batch(tree)
        if not root['expanded']:
            if config.wall_time_seconds == 0:
                budget_cutoff = True
            else:
                _go2_evaluate_nodes(tree, [tree['root']], evaluator, config, game)
        while root['expanded'] and tree['completed_simulations'] < stop:
            elapsed = tree['elapsed_seconds'] + time.monotonic() - started
            if config.wall_time_seconds is not None and elapsed >= config.wall_time_seconds:
                budget_cutoff = True
                break
            # A pause threshold never changes the configured atomic batch shape.
            # It may stop just after the requested simulation count.
            remaining = config.simulations-tree['completed_simulations']
            _go2_collect_batch(tree, config, game, min(config.batch_size, remaining))
            _go2_resolve_batch(tree, evaluator, config, game)
            tree['elapsed_seconds'] += time.monotonic() - started
            started = time.monotonic()
            if on_batch is not None:
                on_batch(tree)
    finally:
        tree['elapsed_seconds'] += time.monotonic()-started
    counts = np.zeros(game['actions'](state), dtype=np.int64)
    for action, edge in root['edges'].items():
        counts[action] = edge['n']
    policy = counts.astype(np.float64)
    if policy.sum():
        policy /= policy.sum()
        action = max(root['edges'], key=lambda a: (root['edges'][a]['n'], root['edges'][a]['prior'], -a))
    else:
        legal = sorted(game['legal'](state))
        action = max(root['edges'], key=lambda a: (root['edges'][a]['prior'], -a)) if root['edges'] else legal[0]
        # No completed simulation means no training policy target.
    status = ('BUDGET_CUTOFF' if budget_cutoff else
              'COMPLETE' if tree['completed_simulations'] >= config.simulations else 'PAUSED')
    root_q = {action: edge['w']/edge['n'] if edge['n'] else None
              for action, edge in root['edges'].items()}
    return {'action': int(action), 'policy_target': policy.tolist(),
            'visit_counts': counts.tolist(), 'tree': tree, 'status': status,
            'search_status': 'UNKNOWN', 'move_legality': 'LEGAL',
            'policy_target_valid': bool(counts.sum()),
            'value': None if root['value'] is None else float(root['value']), 'root_q': root_q,
            'completed_simulations': tree['completed_simulations'],
            'elapsed_seconds': tree['elapsed_seconds'],
            'counters': copy.deepcopy(tree['counters']),
            'queen': copy.deepcopy(tree['queen'])}

## Analytical fixtures

These tiny invented game trees have known outcomes, so they test the search algorithm independently of Go knowledge or network strength. The dangerous move allows an opponent win one move later; the safe move draws. A separate fixture gives a winning move a prior of only 0.1%. The full Go fixtures below test exact pass scoring and history-aware identities.

In [ ]:

def _go2_fixture_state(mode='adversarial', path=()):
    return {'mode': mode, 'path': tuple(path), 'to_play': 1 if len(path)%2 == 0 else -1}


def _go2_fixture_game():
    def legal(state):
        if len(state['path']) >= 2:
            return []
        if state['mode'] == 'low_prior' and state['path']:
            return [0]
        return [0, 1]
    def step(state, action):
        if action not in legal(state):
            raise ValueError('Illegal fixture action.')
        return _go2_fixture_state(state['mode'], (*state['path'], int(action)))
    def value(state):
        first, second = state['path']
        if state['mode'] == 'low_prior':
            winner = -1 if first == 0 else 1
        else:
            winner = (-1 if second == 0 else 1) if first == 0 else 0
        return float(state['to_play'] * winner)
    return {'legal': legal, 'step': step,
            'key': lambda state: state,
            'dump': lambda state: {**state, 'path': list(state['path'])},
            'load': lambda state: _go2_fixture_state(state['mode'], state['path']),
            'terminal': lambda state: len(state['path']) >= 2,
            'value': value, 'actions': lambda state: 2}


def _go2_fixture_eval(states):
    results = []
    for state in states:
        if len(state['path']) >= 2:
            raise AssertionError('Terminal fixture reached the neural evaluator.')
        if state['mode'] == 'low_prior':
            prior = [0.999, 0.001] if not state['path'] else [1.0, 0.0]
        else:
            prior = [0.9, 0.1] if not state['path'] else [0.8, 0.2]
        results.append({'prior': prior, 'value': 0.0, 'network_version': 'analytical-fixture-v1'})
    return results


def _go2_fixture_search(config, tree=None, stop_after_simulations=None, mode='adversarial'):
    return go2_search(None, _go2_fixture_state(mode), config, tree=tree,
                      evaluator=_go2_fixture_eval, model_identity='analytical-fixture-v1',
                      stop_after_simulations=stop_after_simulations, _game=_go2_fixture_game())


def _go2_scientific_tree(tree, game=None):
    payload = go2_search_state_dict(tree, _game=game)
    payload.pop('payload_sha256')
    payload.pop('elapsed_seconds')
    return payload


def go2_search_validation(project=None):
    project = go2_project(project)
    checks = {}
    config = Go2SearchConfig(simulations=512, batch_size=8)
    adversarial = _go2_fixture_search(config)
    checks['adversarial_two_ply_sign'] = bool(adversarial['action'] == 1 and adversarial['root_q'][0] < -0.5)
    low = _go2_fixture_search(Go2SearchConfig(simulations=128, batch_size=8), mode='low_prior')
    checks['low_prior_winner_is_eligible'] = bool(low['action'] == 1 and low['visit_counts'][1] > 0)
    checks['all_legal_edges_exist'] = set(low['tree']['nodes'][0]['edges']) == {0, 1}
    checks['real_visit_conservation'] = bool(sum(adversarial['visit_counts']) == 512 and go2_validate_search_tree(adversarial['tree']))
    checks['no_virtual_visits_in_policy'] = all(node['virtual_visits'] == 0 and all(edge['virtual_n'] == 0 for edge in node['edges'].values())
                                                for node in adversarial['tree']['nodes'].values())
    checks['duplicate_reservations_do_not_double_count'] = bool(adversarial['counters']['duplicate_reservations'] > 0
                                                               and sum(adversarial['visit_counts']) == 512)
    state = go2_initial_state()
    pass_action = state.size * state.size
    one_pass = go2_step(state, pass_action)
    terminal = go2_step(one_pass, pass_action)
    inference_calls = []
    def pass_evaluator(states):
        inference_calls.extend(states)
        outputs = []
        for position in states:
            if position.terminal:
                raise AssertionError('Exact terminal state sent to neural evaluation.')
            prior = np.zeros(position.size*position.size+1)
            prior[-1] = 1.0
            outputs.append({'prior': prior, 'value': 0.0})
        return outputs
    terminal_result = go2_search(None, terminal, Go2SearchConfig(simulations=8), evaluator=pass_evaluator,
                                model_identity='pass-fixture-v1', project=project)
    pass_result = go2_search(None, one_pass, Go2SearchConfig(simulations=16, batch_size=4), evaluator=pass_evaluator,
                            model_identity='pass-fixture-v1', project=project)
    checks['terminal_before_inference'] = terminal_result['counters']['neural_positions'] == 0 and terminal_result['value'] == -1.0
    checks['terminal_pass_parent_sign'] = pass_result['action'] == pass_action and pass_result['root_q'][pass_action] == 1.0
    checks['terminal_exact_white_winner'] = go2_outcome(terminal)['winner'] == -1
    history_a = _go2_inference_key(pass_result['tree'], state, _go2_search_game())
    history_b = _go2_inference_key(pass_result['tree'], one_pass, _go2_search_game())
    context_tree = _go2_new_tree({**pass_result['tree']['identity'], 'context': go2_digest({'queen': 'different'})},
                                state, Go2SearchConfig(simulations=16, batch_size=4), _go2_search_game())
    checks['history_and_queen_context_cache_keys'] = history_a != history_b and history_a != _go2_inference_key(context_tree, state, _go2_search_game())
    zero = go2_search(None, state, Go2SearchConfig(simulations=8, wall_time_seconds=0), evaluator=pass_evaluator,
                     model_identity='pass-fixture-v1', project=project)
    checks['cutoff_unknown_and_legal_separate'] = zero['status'] == 'BUDGET_CUTOFF' and zero['search_status'] == 'UNKNOWN' and zero['move_legality'] == 'LEGAL' and not zero['policy_target_valid']
    shallow = _go2_fixture_search(Go2SearchConfig(simulations=16, batch_size=4, max_depth=1))
    checks['depth_cutoff_remains_unknown'] = shallow['counters']['depth_cutoffs'] > 0 and shallow['search_status'] == 'UNKNOWN'
    configured = Go2SearchConfig(simulations=64, batch_size=8)
    uninterrupted_batch = _go2_fixture_search(configured)
    paused_batch = _go2_fixture_search(configured, stop_after_simulations=32)
    resumed_batch = _go2_fixture_search(configured, tree=paused_batch['tree'])
    checks['pause_preserves_atomic_batch_shape'] = go2_nested_equal(_go2_scientific_tree(uninterrupted_batch['tree'], _go2_fixture_game()), _go2_scientific_tree(resumed_batch['tree'], _go2_fixture_game()))
    # Serialize an actual active reservation batch, not only a finished search.
    partial = _go2_fixture_search(Go2SearchConfig(simulations=64, batch_size=8), stop_after_simulations=32)
    active = partial['tree']
    _go2_collect_batch(active, Go2SearchConfig(simulations=64, batch_size=8), _go2_fixture_game(), 8)
    payload = go2_search_state_dict(active, _game=_go2_fixture_game())
    restored = go2_search_from_state_dict(payload, None, _go2_fixture_state(), model_identity='analytical-fixture-v1',
                                         project=project, _game=_go2_fixture_game())
    pending_clone = copy.deepcopy(active)
    _go2_resolve_batch(pending_clone, _go2_fixture_eval, Go2SearchConfig(simulations=64, batch_size=8), _go2_fixture_game())
    _go2_resolve_batch(restored, _go2_fixture_eval, Go2SearchConfig(simulations=64, batch_size=8), _go2_fixture_game())
    checks['active_pending_batch_exact_restore'] = go2_digest(_go2_scientific_tree(pending_clone, _go2_fixture_game())) == go2_digest(_go2_scientific_tree(restored, _go2_fixture_game()))
    wrong_identity_rejected = False
    try:
        go2_search_from_state_dict(payload, None, _go2_fixture_state(), model_identity='other-model', _game=_go2_fixture_game())
    except ValueError:
        wrong_identity_rejected = True
    checks['wrong_model_identity_rejected'] = wrong_identity_rejected
    corrupted = copy.deepcopy(payload)
    corrupted['completed_simulations'] += 1
    corruption_rejected = False
    try:
        go2_search_from_state_dict(corrupted, None, _go2_fixture_state(), model_identity='analytical-fixture-v1', _game=_go2_fixture_game())
    except ValueError:
        corruption_rejected = True
    checks['corrupt_search_payload_rejected'] = corruption_rejected
    result = {'schema': 'ahr-go-v2-search-validation', 'status': 'PASS' if all(checks.values()) else 'FAIL',
              'checks': checks, 'sources': go2_source_identity(project),
              'fixture_results': {'adversarial_action': adversarial['action'],
                                  'adversarial_root_q': adversarial['root_q'],
                                  'low_prior_winning_visits': low['visit_counts'][1],
                                  'terminal_pass_q': pass_result['root_q'][pass_action],
                                  'duplicate_reservations': adversarial['counters']['duplicate_reservations']},
              'claim': 'Search correctness on bounded fixtures; no learned playing strength established.'}
    if not all(checks.values()):
        raise AssertionError(result)
    return result


## Exact checkpoint recovery

The recovery fixture uses notebook 00's complete-state, atomic, single-writer checkpoint manager. It compares an uninterrupted 64-simulation run against 32 simulations, a saved active reservation batch, and continuation in a fresh Python process. The synthetic checkpoint contains every required training-state slot; optimizer recovery is verified separately in notebook 00.

To resume an application: import only tagged upstream definitions, load the checkpoint with its exact identity using `go2_load_checkpoint(..., restore_rng=True)`, reconstruct `state["search"]` with `go2_search_from_state_dict`, then pass that tree back to `go2_search`. Never reset the simulation cursor or wall-clock budget.

In [ ]:

def go2_search_recovery_validation(project=None):
    import tempfile, subprocess, sys, json
    project = go2_project(project)
    parent = project / 'checkpoints/ahr-v2-search-validation'
    parent.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix='resume-', dir=parent))
    config = Go2SearchConfig(simulations=64, batch_size=2, root_noise_fraction=0.15)
    identity = {'purpose': 'analytical_search_resume', 'sources': go2_source_identity(project),
                'model': 'analytical-fixture-v1', 'config': dataclasses.asdict(config)}
    go2_seed(config.seed)
    uninterrupted = _go2_fixture_search(config)
    baseline_rng = go2_capture_rng()
    go2_seed(config.seed)
    partial = _go2_fixture_search(config, stop_after_simulations=32)
    _go2_collect_batch(partial['tree'], config, _go2_fixture_game(), config.batch_size)
    state = {'schema': GO2_SCHEMA, 'identity': identity,
             'model': {'fixture_model_identity': 'analytical-fixture-v1'},
             'optimizer': None, 'scheduler': None, 'scaler': None,
             'rng': go2_capture_rng(), 'sampler': {'simulation_cursor': 32},
             'global_step': 32, 'config': dataclasses.asdict(config),
             'metrics': {'completed_simulations': 32},
             'search': go2_search_state_dict(partial['tree'], _game=_go2_fixture_game()),
             'replay': [], 'selfplay': None}
    with go2_checkpoint_lock(run_dir):
        go2_save_checkpoint(run_dir, state, best=True,
                            resume_cell='03_batched_puct_search.ipynb: Exact checkpoint recovery')
    program = r"""
import sys, json, dataclasses
from pathlib import Path
import nbformat
project,run_dir=map(Path,sys.argv[1:])
ns={'__name__':'__main__'}
first=nbformat.read(project/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4)
for cell in first.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'00_contracts_and_recovery.ipynb','exec'),ns)
for name in ('01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb'):
    ns['go2_import_exports'](project/'experiments/ahr_v2'/name,ns)
config=ns['Go2SearchConfig'](simulations=64,batch_size=2,root_noise_fraction=0.15)
identity={'purpose':'analytical_search_resume','sources':ns['go2_source_identity'](project),
          'model':'analytical-fixture-v1','config':dataclasses.asdict(config)}
with ns['go2_checkpoint_lock'](run_dir):
    state,entry=ns['go2_load_checkpoint'](run_dir,identity=identity,restore_rng=True)
    tree=ns['go2_search_from_state_dict'](state['search'],None,ns['_go2_fixture_state'](),
          config=config,model_identity='analytical-fixture-v1',project=project,_game=ns['_go2_fixture_game']())
    result=ns['_go2_fixture_search'](config,tree=tree)
    state.update(global_step=64,rng=ns['go2_capture_rng'](),sampler={'simulation_cursor':64},
                 metrics={'completed_simulations':64},
                 search=ns['go2_search_state_dict'](result['tree'],_game=ns['_go2_fixture_game']()))
    ns['go2_save_checkpoint'](run_dir,state,resume_cell='03_batched_puct_search.ipynb: Exact checkpoint recovery')
print(json.dumps({'resumed_from':entry['global_step'],'completed_simulations':result['completed_simulations']}))
"""
    completed = subprocess.run([sys.executable, '-c', program, str(project), str(run_dir)],
                               cwd=project, capture_output=True, text=True, timeout=120)
    if completed.returncode:
        raise RuntimeError(completed.stderr[-5000:])
    resumed_state, entry = go2_load_checkpoint(run_dir, identity=identity)
    resumed_tree = go2_search_from_state_dict(resumed_state['search'], None, _go2_fixture_state(),
                                              config=config, model_identity='analytical-fixture-v1',
                                              project=project, _game=_go2_fixture_game())
    checks = {
        'fresh_process_32_then_64': json.loads(completed.stdout)['resumed_from'] == 32 and entry['global_step'] == 64,
        'entire_scientific_tree_exact': go2_nested_equal(_go2_scientific_tree(uninterrupted['tree'], _go2_fixture_game()),
                                                       _go2_scientific_tree(resumed_tree, _go2_fixture_game())),
        'global_rng_exact': go2_nested_equal(baseline_rng, resumed_state['rng']),
        'active_reservation_batch_restored': bool(state['search']['pending']) and not resumed_tree['pending'],
        'sampler_cursor_exact': resumed_state['sampler']['simulation_cursor'] == 64,
        'latest_checkpoint_step_exact': entry['global_step'] == 64,
        'elapsed_budget_preserved': resumed_tree['elapsed_seconds'] >= partial['tree']['elapsed_seconds'],
        'complete_required_checkpoint_state': GO2_REQUIRED_STATE.issubset(resumed_state),
    }
    result = {'schema': 'ahr-go-v2-search-recovery-validation', 'status': 'PASS' if all(checks.values()) else 'FAIL',
              'checks': checks, 'sources': go2_source_identity(project), 'run_dir': str(run_dir.relative_to(project)),
              'completed_simulations': resumed_tree['completed_simulations'],
              'claim': 'Exact CPU search recovery, including an active inference batch; no training or strength claim.'}
    if not all(checks.values()):
        raise AssertionError(result)
    return result


## Run the CPU checks

A fresh network is tested on two exact Go positions. The network is untrained, so these checks establish legal integration and completed-visit accounting only. The fixture tables report bounded correctness checks; they do not report a win rate.

In [ ]:
import torch
import pandas as pd
from IPython.display import display

torch.set_num_threads(2)
go2_search_checks = go2_search_validation(GO2_PROJECT)
go2_search_recovery = go2_search_recovery_validation(GO2_PROJECT)

go2_seed(2026100603)
go2_smoke_model = Go2Network(Go2NetConfig()).cpu().eval()
go2_smoke_positions = [go2_initial_state()]
go2_midgame = go2_initial_state()
for _ in range(8):
    go2_midgame = go2_step(go2_midgame, next(action for action in go2_legal_actions(go2_midgame) if action != 81))
go2_smoke_positions.append(go2_midgame)
go2_smoke_results = []
for _index, _state in enumerate(go2_smoke_positions):
    _result = go2_search(go2_smoke_model, _state, Go2SearchConfig(simulations=16, batch_size=8), project=GO2_PROJECT)
    assert _result['action'] in go2_legal_actions(_state)
    assert sum(_result['visit_counts']) == 16
    assert set(_result['tree']['nodes'][0]['edges']) == set(go2_legal_actions(_state))
    go2_smoke_results.append({'position': 'opening' if _index == 0 else 'eight legal moves',
                             'selected_action': _result['action'],
                             'completed_simulations': _result['completed_simulations'],
                             'legal_root_actions': len(_result['tree']['nodes'][0]['edges']),
                             'neural_positions': _result['counters']['neural_positions'],
                             'status': _result['status'], 'game_claim': _result['search_status']})

go2_search_receipt = {'schema': 'ahr-go-v2-search-delivery', 'status': 'PASS',
                      'sources': go2_source_identity(GO2_PROJECT),
                      'algorithm': go2_search_checks, 'recovery': go2_search_recovery,
                      'fresh_untrained_network': go2_smoke_results,
                      'created_at_utc': go2_now()}
go2_report_dir = GO2_PROJECT / 'reports/ahr_v2'
go2_report_dir.mkdir(parents=True, exist_ok=True)
go2_atomic_json(go2_report_dir / 'search_validation.json', go2_search_receipt)
display(pd.DataFrame([{'check': key.replace('_', ' '), 'passed': value}
                      for key, value in go2_search_checks['checks'].items()]))
display(pd.DataFrame([{'recovery check': key.replace('_', ' '), 'passed': value}
                      for key, value in go2_search_recovery['checks'].items()]))
display(pd.DataFrame(go2_smoke_results))
print('PASS: exact rule integration, adversarial backup, all legal actions and fresh-process search recovery.')

## What this establishes

The saved checks verify the search's value signs, legal action eligibility, exact terminal handling, completed-visit targets and interruption recovery. They also verify that history and Queen context participate in cache identities. The next training stage can use this search with completed-game outcome targets, without requiring a tactical pretraining gate.

The Queen currently follows a fixed batch schedule. No learned Queen, relativistic workspace, teacher dataset, training run or strong-player claim is included here.